# Building a Data Pipeline 

### Imports

In [44]:
import numpy as np 
import pandas as pd 
import random
from randomtimestamp import randomtimestamp # type: ignore
from geopy.geocoders import Nominatim # type: ignore
from meteostat import Point, Daily # type: ignore
from datetime import datetime, timedelta
import time

import seaborn as sns 
import matplotlib.pyplot as plt 

from sklearn import preprocessing, svm 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression 
from sklearn.preprocessing import OneHotEncoder


df_import = pd.read_csv("import_data/new_dummy_data.csv", delimiter=";")

### Data Cleaning

# 1. Data Acquisition

In [45]:
df_import.head(5)

,date,flight_number,departure,destination,travel_class,service,component,dish_type,region,passengers,...,price_per_dish,co2_per_item,items_saved,potential_cost_savings,potential_co2_savings,customer_satisfaction,departure_time,arrival_time,flight_duration,route
0,2024-04-19,4Y001,FRA,JFK,Business,First,Starter,Meat,North America,35,...,5,"2,5",3.0,15,"7,5",84.0,08:00,11:00,08:00,intercont
1,2024-04-19,4Y001,FRA,JFK,Business,First,Starter,Veg,North America,35,...,"4,5","1,2",3.0,"13,5","3,6",100.0,08:00,11:00,08:00,intercont
2,2024-04-19,4Y001,FRA,JFK,Business,First,Main,Meat,North America,35,...,5,"2,5",0.0,0,0,93.0,08:00,11:00,08:00,intercont
3,2024-04-19,4Y001,FRA,JFK,Business,First,Main,Veg,North America,35,...,"4,5","1,2",2.0,9,"2,4",91.0,08:00,11:00,08:00,intercont
4,2024-04-19,4Y001,FRA,JFK,Business,First,Main,Fish,North America,35,...,6,3,0.0,0,0,85.0,08:00,11:00,08:00,intercont


## 1.1 Add Geolocation

In [46]:
geolocator = Nominatim(user_agent="iata_locator")
def get_airport_coordinates(iata_code):
    time.sleep(0.1)
    location = geolocator.geocode(f"{iata_code} airport")
    if location:
        return {
            "latitude": location.latitude,
            "longitude": location.longitude,
        }
    else:
        return None
    

def create_df(cities):
    # requires an unique set of cities formatted as an array
    length = len(cities)
    coords = []
    for i in range(0, length):                
        coords.append(get_airport_coordinates(cities[i]))
        time.sleep(0.5)
            
    print(f"Anzahl der Städte: {len(coords)}")

    # DataFrame mit Koordinaten
    df_coords = pd.DataFrame({
        'city': cities,
        'latitude': [x['latitude'] for x in coords],
        'longitude': [x['longitude'] for x in coords]
    })
    return df_coords

departure_airports = set(df_import["departure"])
destination_airports = set(df_import["destination"])

all_airports = list(departure_airports.union(destination_airports))

df_airports = create_df(all_airports)

df_import['departure'] = df_import['departure'].astype(str)
df_import['destination'] = df_import['destination'].astype(str)
df_airports['city'] = df_airports['city'].astype(str)

df = df_import.merge(df_airports.rename(columns={
    'city': 'departure',
    'latitude': 'departure_latitude',
    'longitude': 'departure_longitude'
}), on='departure', how='left')

df = df.merge(df_airports.rename(columns={
    'city': 'destination',
    'latitude': 'destination_latitude',
    'longitude': 'destination_longitude'
}), on='destination', how='left')

df.head()


Anzahl der Städte: 13


,date,flight_number,departure,destination,travel_class,service,component,dish_type,region,passengers,...,potential_co2_savings,customer_satisfaction,departure_time,arrival_time,flight_duration,route,departure_latitude,departure_longitude,destination_latitude,destination_longitude
0,2024-04-19,4Y001,FRA,JFK,Business,First,Starter,Meat,North America,35,...,"7,5",84.0,08:00,11:00,08:00,intercont,50.024413,8.5552,40.642948,-73.779373
1,2024-04-19,4Y001,FRA,JFK,Business,First,Starter,Veg,North America,35,...,"3,6",100.0,08:00,11:00,08:00,intercont,50.024413,8.5552,40.642948,-73.779373
2,2024-04-19,4Y001,FRA,JFK,Business,First,Main,Meat,North America,35,...,0,93.0,08:00,11:00,08:00,intercont,50.024413,8.5552,40.642948,-73.779373
3,2024-04-19,4Y001,FRA,JFK,Business,First,Main,Veg,North America,35,...,"2,4",91.0,08:00,11:00,08:00,intercont,50.024413,8.5552,40.642948,-73.779373
4,2024-04-19,4Y001,FRA,JFK,Business,First,Main,Fish,North America,35,...,0,85.0,08:00,11:00,08:00,intercont,50.024413,8.5552,40.642948,-73.779373


## 1.2 Add Weather Data

In [40]:
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')
df['date'] = df.to_string('date')

In [47]:
df.dtypes

date                        object
flight_number               object
departure                   object
destination                 object
travel_class                object
service                     object
component                   object
dish_type                   object
region                      object
passengers                   int64
amount_loaded                int64
amount_predicted_demand      int64
amount_used                float64
amount_missed_order        float64
recommended_ratio           object
ratio                       object
load_factor                 object
price_per_dish              object
co2_per_item                object
items_saved                float64
potential_cost_savings      object
potential_co2_savings       object
customer_satisfaction      float64
departure_time              object
arrival_time                object
flight_duration             object
route                       object
departure_latitude         float64
departure_longitude 

In [59]:
def get_weather(lat, lon, date):

    key = (lat, lon, date)
    today = datetime.today()

    date_no_string = datetime.strptime(date, '%Y-%m-%d')

    if today > date_no_string:
        today = today - timedelta(days=365)
    if key in weather_cache:
        return weather_cache[key]
    else:
        location = Point(lat, lon)

        start_date = end_date = datetime.strptime(date, '%Y-%m-%d')
        data = Daily(location, start_date, end_date).fetch()

        weather_columns = ['tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun']

        if not data.empty:
            weather_data = {col: data[col].iloc[0] if col in data.columns else None for col in weather_columns}
        else:
            weather_data = {col: None for col in weather_columns}

        weather_cache[key] = weather_data
        time.sleep(0.1)
        return weather_data

weather_cache = {}
weather_results = []

for index, row in df.head(50).iterrows():
    try:
        departure_weather = get_weather(row['departure_latitude'], row['departure_longitude'], row['date'])
        
        departure_datetime = datetime.strptime(f"{row['date']} {row['departure_time']}", "%Y-%m-%d %H:%M")
        arrival_time = datetime.strptime(row['arrival_time'], "%H:%M").time()
        arrival_datetime = datetime.combine(departure_datetime.date(), arrival_time)

        if arrival_datetime < departure_datetime:
            arrival_datetime += timedelta(days=1)

        arrival_date = arrival_datetime.strftime("%Y-%m-%d")

        destination_weather = get_weather(row['destination_latitude'], row['destination_longitude'], arrival_date)

        combined_weather = {
            **{f"departure_{key}": value for key, value in departure_weather.items()},
            **{f"destination_{key}": value for key, value in destination_weather.items()}
        }
        weather_results.append(combined_weather)

    except Exception as e:
        print(f"Fehler in Zeile {index}: {e}")
        weather_results.append({col: None for col in combined_weather.keys()})

weather_df = pd.DataFrame(weather_results)
df = pd.concat([df, weather_df], axis=1)


In [28]:
# === Save Weather Data ===
df.to_csv('output/weather_data_output.csv', index=False)

## 1.3 Add Country Data

In [29]:
country_cache = {}

country_results = []
def get_country(lat, lon):

    key = (lat, lon)
    
    if key in country_cache:
        return country_cache[key]
    else:
        try:
            location = geolocator.reverse((lat, lon), exactly_one=True, language="en")
            country = location.raw.get('address', {}).get('country', None)
            country_cache[key] = country
            time.sleep(0.3) 
            return country
        except Exception as e:
            print(f"Fehler beim Abrufen des Landes für {key}: {e}")
            return None
        
for index, row in df.iterrows():
    if index >= 50:
        break
    try: 
        departure_country = get_country(row['departure_latitude'], row['departure_longitude'])
        destination_country = get_country(row['destination_latitude'], row['destination_longitude'])
        
        combined_countries = {
            "departure_country": departure_country,
            "destination_country": destination_country
        }
    except:
        combined_countries = {
            "departure_country": None,
            "destination_country": None
        }
    
    country_results.append(combined_countries)

country_df = pd.DataFrame(country_results)
df = pd.concat([df, country_df], axis=1)

# ISO-Codes für Staaten
country_codes = pd.read_csv(r'import_data/country_codes.csv')
print (country_codes)
df['departure_country_code'] = ''
df['destination_country_code'] = ''
df = df.merge(country_codes, left_on='departure_country', right_on='Name', how='left')
df['departure_country_code'] = df['Code']
df = df.drop({'Name', 'Code'}, axis = 1)

df = df.merge(country_codes, left_on='destination_country', right_on='Name', how='left')
df['destination_country_code'] = df['Code']
df = df.drop({'Name', 'Code'}, axis = 1)


               Name Code
0       Afghanistan   AF
1           Albania   AL
2           Algeria   DZ
3    American Samoa   AS
4           Andorra   AD
..              ...  ...
244  Western Sahara   EH
245           Yemen   YE
246          Zambia   ZM
247        Zimbabwe   ZW
248   Åland Islands   AX

[249 rows x 2 columns]


## 1.4 Add GDP / Population 

In [30]:
#BIP für Staaten
import requests

gdp_cache = {}
population_cache = {}

df['departure_GDP_per_capita'] = ''
df['destination_GDP_per_capita'] = ''

def get_data_from_worldbank(country_code, indicator):
    """
    Holt Daten von der World Bank API für einen bestimmten Länder-Code und Indikator.
    """
    key = (country_code, indicator)

    if indicator == 'NY.GDP.MKTP.CD' and country_code in gdp_cache:
        return gdp_cache[country_code]
    if indicator == 'SP.POP.TOTL' and country_code in population_cache:
        return population_cache[country_code]
    
    try:
        url = f'http://api.worldbank.org/v2/country/{country_code}/indicator/{indicator}?format=json'
        response = requests.get(url)
        data = response.json()
        
        if response.status_code == 200 and len(data) > 1 and data[1]:
            value = data[1][0].get('value', None)
        else:
            value = None

        if indicator == 'NY.GDP.MKTP.CD':
            gdp_cache[country_code] = value
        elif indicator == 'SP.POP.TOTL':
            population_cache[country_code] = value

        time.sleep(0.1)
        return value
    except Exception as e:
        print(f"Fehler beim Abrufen von {indicator} für {country_code}: {e}")
        if indicator == 'NY.GDP.MKTP.CD':
            gdp_cache[country_code] = None
        elif indicator == 'SP.POP.TOTL':
            population_cache[country_code] = None
        return None

def calculate_gdp_per_capita(country_code):
    
    gdp = get_data_from_worldbank(country_code, 'NY.GDP.MKTP.CD')
    population = get_data_from_worldbank(country_code, 'SP.POP.TOTL')

    if gdp is not None and population is not None and population > 0:
        return gdp / population
    return None

for index, row in df.iterrows():
    if index >= 50:
        break
    try:
        df.at[index, 'departure_GDP_per_capita'] = calculate_gdp_per_capita(row['departure_country_code'])
        df.at[index, 'destination_GDP_per_capita'] = calculate_gdp_per_capita(row['destination_country_code'])
    except Exception as e:
        print(f"Fehler in Zeile {index}: {e}")

## 1.5 World-wide public Holidays

In [8]:
def get_public_holidays(year, country_code, holiday_cache):
    if (year, country_code) in holiday_cache:
        return holiday_cache[(year, country_code)]
    
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            holidays = response.json()
            holiday_dates = {holiday["date"] for holiday in holidays}
            holiday_cache[(year, country_code)] = holiday_dates
            time.sleep(0.1)
            return holiday_dates
        else:
            print(f"Fehler bei der Anfrage für {country_code}: {response.status_code}")
            holiday_cache[(year, country_code)] = set()
            return set()
    except Exception as e:
        print(f"Fehler beim Abrufen der Daten für {country_code}: {e}")
        holiday_cache[(year, country_code)] = set()
        return set()

def add_holiday_columns(df):
    holiday_cache = {}
    years = []
    for date in df['date']:
        year = pd.to_datetime(date, format='%Y-%m-%d').year
        years.append(year)
        
    country_codes = set(df["departure_country_code"].dropna()).union(set(df["destination_country_code"].dropna()))
    
    for year in years:
        for country_code in country_codes:
            get_public_holidays(year, country_code, holiday_cache)
    
    df["departure_holiday"] = False
    df["destination_holiday"] = False
    
    for index, row in df.iterrows():
        if index > 50:
            break
        year = row["date"][:4]
        departure_code = row["departure_country_code"]
        destination_code = row["destination_country_code"]
        date = row["date"]
        
        if pd.notna(departure_code):
            df.at[index, "departure_holiday"] = date in holiday_cache.get((year, departure_code), set())
        
        if pd.notna(destination_code):
            df.at[index, "destination_holiday"] = date in holiday_cache.get((year, destination_code), set())
    
    return df


df = add_holiday_columns(df)

## ML Preprocessing

In [14]:
print(df.loc[df['departure_holiday'] == True])


Empty DataFrame
Columns: [date, flight_number, departure, destination, travel_class, service, component, dish_type, region, passengers, amount_loaded, amount_predicted_demand, amount_used, amount_missed_order, recommended_ratio, ratio, load_factor, price_per_dish, co2_per_item, items_saved, potential_cost_savings, potential_co2_savings, customer_satisfaction, departure_time, arrival_time, flight_duration, route, departure_latitude, departure_longitude, destination_latitude, destination_longitude, departure_tavg, departure_tmin, departure_tmax, departure_prcp, departure_snow, departure_wdir, departure_wspd, departure_wpgt, departure_pres, departure_tsun, destination_tavg, destination_tmin, destination_tmax, destination_prcp, destination_snow, destination_wdir, destination_wspd, destination_wpgt, destination_pres, destination_tsun, departure_country, destination_country, departure_country_code, destination_country_code, departure_GDP_per_capita, destination_GDP_per_capita, departure_holi

In [10]:
print(list(df.columns))


['date', 'flight_number', 'departure', 'destination', 'travel_class', 'service', 'component', 'dish_type', 'region', 'passengers', 'amount_loaded', 'amount_predicted_demand', 'amount_used', 'amount_missed_order', 'recommended_ratio', 'ratio', 'load_factor', 'price_per_dish', 'co2_per_item', 'items_saved', 'potential_cost_savings', 'potential_co2_savings', 'customer_satisfaction', 'departure_time', 'arrival_time', 'flight_duration', 'route', 'departure_latitude', 'departure_longitude', 'destination_latitude', 'destination_longitude', 'departure_tavg', 'departure_tmin', 'departure_tmax', 'departure_prcp', 'departure_snow', 'departure_wdir', 'departure_wspd', 'departure_wpgt', 'departure_pres', 'departure_tsun', 'destination_tavg', 'destination_tmin', 'destination_tmax', 'destination_prcp', 'destination_snow', 'destination_wdir', 'destination_wspd', 'destination_wpgt', 'destination_pres', 'destination_tsun', 'departure_country', 'destination_country', 'departure_country_code', 'destinatio